In [2]:
# Install dependencies

import subprocess
subprocess.run([
    "pip", "install", "-q",
    "langchain",
    "langchain-experimental",
    "langchain-huggingface",
    "sentence-transformers"
], check=True)

print("✓ Dependencies installed")

✓ Dependencies installed


In [4]:
# Imports

import os
import pickle
import time
import warnings
warnings.filterwarnings("ignore")

# Updated imports (new LangChain structure)
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)

from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

print("✓ Imports done")

✓ Imports done


In [18]:
# Configuration

CLEANED_DIR = "/kaggle/input/datasets/rockybhai159/cleaned/Cleaned"

OUTPUT_DIR  = "/kaggle/working"   # Kaggle saves here — do not change

# Verify cleaned dir exists and has files
txt_files = sorted([f for f in os.listdir(CLEANED_DIR) if f.endswith('.txt')])
print(f"✓ Found {len(txt_files)} cleaned .txt files in {CLEANED_DIR}")
for f in txt_files:
    size = os.path.getsize(os.path.join(CLEANED_DIR, f))
    print(f"   {f:55s}  {size/1024:.1f} KB")

✓ Found 15 cleaned .txt files in /kaggle/input/datasets/rockybhai159/cleaned/Cleaned
   00_nested_learning_original.txt                          176.0 KB
   03_fast_weight_programming.txt                           86.6 KB
   04_end_to_end_ttt_long_context.txt                       73.2 KB
   05_dynamic_nested_hierarchies.txt                        34.4 KB
   06_crad_hope_anomaly_detection.txt                       59.4 KB
   08_tiny_autoregressive_recursive.txt                     32.4 KB
   10_hydra_dual_exponentiated_memory.txt                   38.6 KB
   11_beyond_ttt_optimal_control.txt                        57.8 KB
   13_allmem_long_context.txt                               36.2 KB
   14_vision_transformers_never_stop.txt                    33.8 KB
   15_msa_memory_sparse_attention.txt                       55.0 KB
   16_prompt_injection_nested_learning.txt                  79.2 KB
   17_memory_caching_rnns.txt                               50.8 KB
   22_learning_to_learn_at_sca

In [19]:
# Paper metadata

PAPER_META = {
    "00_nested_learning_original": {
        "title":    "Nested Learning: The Illusion of Deep Learning Architectures",
        "year":     2025,
        "category": "Core Extension",
        "authors":  "Behrouz, Razaviyayn, Zhong, Mirrokni",
    },
    "03_fast_weight_programming": {
        "title":    "Fast Weight Programming and Linear Transformers: From ML to Neurobiology",
        "year":     2026,
        "category": "Related Theory",
        "authors":  "Irie, Gershman",
    },
    "04_end_to_end_ttt_long_context": {
        "title":    "End-to-End Test-Time Training for Long Context",
        "year":     2025,
        "category": "Core Extension",
        "authors":  "Tandon, Dalal, Li et al.",
    },
    "05_dynamic_nested_hierarchies": {
        "title":    "Dynamic Nested Hierarchies: Self-Evolution in ML for Lifelong Intelligence",
        "year":     2025,
        "category": "Core Extension",
        "authors":  "Akbar Anbar Jafari, Ozcinar, Anbarjafari",
    },
    "06_crad_hope_anomaly_detection": {
        "title":    "CRAD-HOPE: Brain-inspired Nested Learning Framework for Few-Shot Anomaly Detection",
        "year":     2026,
        "category": "Application",
        "authors":  "Cao, Cao, Wu",
    },
    "08_tiny_autoregressive_recursive": {
        "title":    "Tiny Autoregressive Recursive Models",
        "year":     2026,
        "category": "Related Theory",
        "authors":  "Rauba, Fanconi, van der Schaar",
    },
    "10_hydra_dual_exponentiated_memory": {
        "title":    "HYDRA: Dual Exponentiated Memory for Multivariate Time Series Analysis",
        "year":     2025,
        "category": "Core Extension",
        "authors":  "Meskin, Mirrokni, Najar, Behrouz",
    },
    "11_beyond_ttt_optimal_control": {
        "title":    "Beyond Test-Time Training: Learning to Reason via Hardware-Efficient Optimal Control",
        "year":     2026,
        "category": "Core Extension",
        "authors":  "Wang, Yang, Wang, Xiao, Liu et al.",
    },
    "13_allmem_long_context": {
        "title":    "AllMem: A Memory-centric Recipe for Efficient Long-context Modeling",
        "year":     2026,
        "category": "Application",
        "authors":  "Wang, Wang, Peng, Qin et al.",
    },
    "14_vision_transformers_never_stop": {
        "title":    "Vision Transformers That Never Stop Learning",
        "year":     2026,
        "category": "Application",
        "authors":  "Sun, Yuan, Wang, Chen",
    },
    "15_msa_memory_sparse_attention": {
        "title":    "MSA: Memory Sparse Attention for Efficient End-to-End Memory Model Scaling",
        "year":     2026,
        "category": "Application",
        "authors":  "Chen, Chen, Yi, Zhao et al.",
    },
    "16_prompt_injection_nested_learning": {
        "title":    "Prompt Injection Mitigation with Agentic AI, Nested Learning, and AI Sustainability",
        "year":     2026,
        "category": "Application",
        "authors":  "Gosmar, Dah",
    },
    "17_memory_caching_rnns": {
        "title":    "Memory Caching: RNNs with Growing Memory",
        "year":     2026,
        "category": "Core Extension",
        "authors":  "Behrouz, Li, Deng, Zhong, Razaviyayn, Mirrokni",
    },
    "22_learning_to_learn_at_scale": {
        "title":    "Learning-to-Learn at Scale: Promises and Challenges",
        "year":     2026,
        "category": "Related Theory",
        "authors":  "Wang, Wang, Chen, Yu, Gan, Liu",
    },
    "26_attention_not_retention": {
        "title":    "Attention Is Not Retention: The Orthogonality Constraint in Infinite-Context Architectures",
        "year":     2025,
        "category": "Related Theory",
        "authors":  "Zahn, Beton, Chana",
    },
}

print(f"✓ Metadata loaded for {len(PAPER_META)} papers")

✓ Metadata loaded for 15 papers


In [25]:
# Define the three splitters

# --- Strategy A: Fixed Character Splitting ---
# Cuts at exactly 512 characters on newline boundaries
# No awareness of sentence or paragraph structure
# Fast but produces lowest quality chunks — used as ablation baseline
splitter_A = CharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    separator="\n",
    length_function=len,
)

# --- Strategy B: Recursive Character Splitting ---
# Tries \n\n → \n → ". " → " " in order, picks nearest natural boundary
# Preserves paragraph and sentence structure wherever possible
# Best balance of quality + speed — used as production system
splitter_B = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " "],
    length_function=len,
)

# --- Strategy C: Semantic Chunking ---
# Embeds every sentence using all-MiniLM-L6-v2
# Splits where cosine similarity between adjacent sentences drops sharply
# Most intelligent — cuts only when the topic genuinely changes
# Slowest — takes 10-20 min on CPU across 15 papers
print("Loading embedding model for semantic chunking (this takes ~2 min)...")
t0 = time.time()
embed_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"batch_size": 64},
)
print(f"✓ Embedding model loaded in {time.time()-t0:.1f}s")

splitter_C = SemanticChunker(
    embeddings=embed_model,
    breakpoint_threshold_type="percentile",
    # Splits where similarity drops below the 70th percentile
    # Lower value = fewer, larger chunks
    # Higher value = more, smaller chunks
    breakpoint_threshold_amount=70,
)

print("✓ All three splitters ready")

Loading embedding model for semantic chunking (this takes ~2 min)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Embedding model loaded in 1.7s
✓ All three splitters ready


In [26]:
# Core build function

def build_corpus(splitter, label, description, min_chunk_len=80):
    """
    Chunks all cleaned .txt files using the given splitter,
    attaches metadata to each chunk, saves as a pkl file.

    min_chunk_len: drops any chunk shorter than this (catches
    micro-chunks from abbreviation mis-splits in semantic mode)
    """
    all_chunks   = []
    all_metadata = []
    per_paper    = {}   # for the summary report

    files = sorted([f for f in os.listdir(CLEANED_DIR) if f.endswith('.txt')])

    print(f"\n{'='*65}")
    print(f"  Strategy {label}: {description}")
    print(f"{'='*65}")

    t_start = time.time()

    for fname in files:
        key       = fname.replace('.txt', '')
        meta_base = PAPER_META.get(key, {
            "title":    key,
            "year":     "unknown",
            "category": "unknown",
            "authors":  "unknown",
        })

        fpath = os.path.join(CLEANED_DIR, fname)
        with open(fpath, 'r', encoding='utf-8') as f:
            text = f.read()

        # Run the splitter
        t0     = time.time()
        chunks = splitter.split_text(text)
        t1     = time.time()

        # Drop micro-chunks (< min_chunk_len chars)
        # Mostly affects Semantic strategy due to abbreviation mis-splits
        before_filter = len(chunks)
        chunks = [c.strip() for c in chunks if len(c.strip()) >= min_chunk_len]
        dropped = before_filter - len(chunks)

        for i, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            all_metadata.append({
                "source":      fname,
                "title":       meta_base["title"],
                "authors":     meta_base["authors"],
                "year":        meta_base["year"],
                "category":    meta_base["category"],
                "chunk_id":    i,
                "chunk_total": len(chunks),
                "strategy":    label,
            })

        avg_len = int(sum(len(c) for c in chunks) / len(chunks)) if chunks else 0
        per_paper[fname] = {
            "chunks": len(chunks),
            "dropped": dropped,
            "avg_len": avg_len,
            "time": round(t1 - t0, 2),
        }

        drop_note = f"  ({dropped} micro-chunks dropped)" if dropped > 0 else ""
        print(f"  {fname:52s}  {len(chunks):3d} chunks  avg={avg_len}c  {t1-t0:.1f}s{drop_note}")

    # ── Save pkl ──────────────────────────────────────────────────────────────
    output_path = os.path.join(OUTPUT_DIR, f"corpus_{label}.pkl")
    with open(output_path, 'wb') as f:
        pickle.dump({
            "chunks":      all_chunks,
            "metadata":    all_metadata,
            "strategy":    label,
            "description": description,
        }, f)

    total_chars = sum(len(c) for c in all_chunks)
    avg_overall = total_chars // len(all_chunks) if all_chunks else 0
    elapsed     = time.time() - t_start
    file_size   = os.path.getsize(output_path) / 1024

    print(f"\n  ✓ Total chunks   : {len(all_chunks)}")
    print(f"  ✓ Avg chunk size : {avg_overall} characters")
    print(f"  ✓ Total time     : {elapsed:.1f}s")
    print(f"  ✓ Saved          : corpus_{label}.pkl  ({file_size:.1f} KB)")

    return {
        "label":       label,
        "description": description,
        "total_chunks": len(all_chunks),
        "avg_chars":   avg_overall,
        "elapsed":     round(elapsed, 1),
        "file_size_kb": round(file_size, 1),
        "per_paper":   per_paper,
    }

print('done')

done


In [27]:
# Run all three strategies


print("Starting chunking pipeline...\n")

results = {}

results["A"] = build_corpus(
    splitter_A,
    label="A_fixed",
    description="Fixed Character Splitting  (512 chars / 50 overlap)",
)

results["B"] = build_corpus(
    splitter_B,
    label="B_recursive",
    description="Recursive Character Splitting  (400 chars / 80 overlap)",
)

results["C"] = build_corpus(
    splitter_C,
    label="C_semantic",
    description="Semantic Chunking  (percentile=70 / all-MiniLM-L6-v2)",
)

Starting chunking pipeline...


  Strategy A_fixed: Fixed Character Splitting  (512 chars / 50 overlap)
  00_nested_learning_original.txt                       364 chunks  avg=467c  0.0s
  03_fast_weight_programming.txt                        189 chunks  avg=459c  0.0s
  04_end_to_end_ttt_long_context.txt                    157 chunks  avg=470c  0.0s
  05_dynamic_nested_hierarchies.txt                      74 chunks  avg=467c  0.0s
  06_crad_hope_anomaly_detection.txt                    126 chunks  avg=472c  0.0s
  08_tiny_autoregressive_recursive.txt                   70 chunks  avg=473c  0.0s
  10_hydra_dual_exponentiated_memory.txt                 79 chunks  avg=486c  0.0s
  11_beyond_ttt_optimal_control.txt                     118 chunks  avg=465c  0.0s
  13_allmem_long_context.txt                             77 chunks  avg=466c  0.0s
  14_vision_transformers_never_stop.txt                  75 chunks  avg=456c  0.0s
  15_msa_memory_sparse_attention.txt                    118 chunks

In [28]:
# Print final comparison summary

print(f"\n{'='*65}")
print(f"  CHUNKING COMPLETE — FINAL SUMMARY")
print(f"{'='*65}")
print(f"  {'Strategy':<40} {'Chunks':>8}  {'Avg Size':>10}  {'Time':>8}  {'File':>8}")
print(f"  {'-'*65}")

for key in ["A","B","C"]:
    r = results[key]
    print(
        f"  {r['label'] + ' — ' + r['description'][:28]:<40}"
        f"  {r['total_chunks']:>6}"
        f"  {r['avg_chars']:>8}c"
        f"  {r['elapsed']:>6}s"
        f"  {r['file_size_kb']:>5}KB"
    )

print(f"\n  Production file  →  corpus_B_recursive.pkl  (upload this to HF Space)")
print(f"  Ablation files   →  corpus_A_fixed.pkl, corpus_C_semantic.pkl")


  CHUNKING COMPLETE — FINAL SUMMARY
  Strategy                                   Chunks    Avg Size      Time      File
  -----------------------------------------------------------------
  A_fixed — Fixed Character Splitting  (      2007       467c     0.1s  1100.9KB
  B_recursive — Recursive Character Splittin    2753       354c     0.1s  1201.3KB
  C_semantic — Semantic Chunking  (percenti    1455       625c    10.4s  1025.5KB

  Production file  →  corpus_B_recursive.pkl  (upload this to HF Space)
  Ablation files   →  corpus_A_fixed.pkl, corpus_C_semantic.pkl


In [29]:
# Save summary report as .txt 

summary_path = os.path.join(OUTPUT_DIR, "chunking_summary.txt")
with open(summary_path, 'w') as f:
    f.write("CHUNKING STRATEGY COMPARISON\n")
    f.write("NLP with Deep Learning — Assignment 3\n")
    f.write("="*65 + "\n\n")

    for key in ["A","B","C"]:
        r = results[key]
        f.write(f"Strategy {r['label']}: {r['description']}\n")
        f.write(f"  Total chunks   : {r['total_chunks']}\n")
        f.write(f"  Avg chunk size : {r['avg_chars']} characters\n")
        f.write(f"  Processing time: {r['elapsed']}s\n")
        f.write(f"  Output file    : corpus_{r['label']}.pkl  ({r['file_size_kb']} KB)\n")
        f.write(f"\n  Per-paper breakdown:\n")
        for fname, stats in r['per_paper'].items():
            f.write(f"    {fname:50s}  {stats['chunks']:3d} chunks  avg={stats['avg_len']}c\n")
        f.write("\n")

print(f"\n  ✓ Summary report saved to chunking_summary.txt")



  ✓ Summary report saved to chunking_summary.txt


In [30]:
# Quick sanity check on all three pkl files

print(f"\n{'='*65}")
print(f"  SANITY CHECK — verifying pkl files")
print(f"{'='*65}")

for label in ["A_fixed", "B_recursive", "C_semantic"]:
    path = os.path.join(OUTPUT_DIR, f"corpus_{label}.pkl")
    with open(path, 'rb') as f:
        data = pickle.load(f)

    chunks   = data["chunks"]
    metadata = data["metadata"]

    # Pick the 10th chunk as sample
    sample_idx = min(10, len(chunks)-1)
    sample_chunk = chunks[sample_idx]
    sample_meta  = metadata[sample_idx]

    print(f"\n  corpus_{label}.pkl")
    print(f"    Chunks    : {len(chunks)}")
    print(f"    Metadata  : {len(metadata)}")
    print(f"    Sample #{sample_idx}:")
    print(f"      Title   : {sample_meta['title'][:60]}")
    print(f"      Category: {sample_meta['category']}")
    print(f"      Chunk   : {sample_meta['chunk_id']} of {sample_meta['chunk_total']}")
    print(f"      Text    : {sample_chunk[:120].strip()}...")

print(f"\n{'='*65}")
print(f"  ALL DONE. Download your pkl files from /kaggle/working/")
print(f"{'='*65}")


  SANITY CHECK — verifying pkl files

  corpus_A_fixed.pkl
    Chunks    : 2007
    Metadata  : 2007
    Sample #10:
      Title   : Nested Learning: The Illusion of Deep Learning Architectures
      Category: Core Extension
      Chunk   : 10 of 364
      Text    : §A version of this work is published at Neural Information Processing Systems (NeurIPS) 2025.
arXiv:2512.24695v1  [cs.LG...

  corpus_B_recursive.pkl
    Chunks    : 2753
    Metadata  : 2753
    Sample #10:
      Title   : Nested Learning: The Illusion of Deep Learning Architectures
      Category: Core Extension
      Chunk   : 10 of 495
      Text    : understanding (Achiam et al. 2023; Liu et al. 2024a; Comanici et al. 2025).
Stacking of multiple layers, as it is done i...

  corpus_C_semantic.pkl
    Chunks    : 1455
    Metadata  : 1455
    Sample #10:
      Title   : Nested Learning: The Illusion of Deep Learning Architectures
      Category: Core Extension
      Chunk   : 10 of 244
      Text    : 2020; Jordan et a